# Data Challenge 10 — MLR Interpretation with Adjusted R² (HVFHV Trips)


**Format:** Instructor Guidance → You Do (Students) → We Share (Reflection)

**Goal:** Build **3 MLR models** with different feature sets to predict a numeric target, then compare **Adjusted R²** and **p-values** to select the better model and justify it in business terms.

**Data:** July 1, 2023 - July 15, 2023 For Hire Vehicle Data in NYC

[July For Hire Vehicles Data](https://data.cityofnewyork.us/Transportation/2023-High-Volume-FHV-Trip-Data/u253-aew4/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (quick links):**
- TLC HVFHV data dictionary (columns/meaning): https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_hvfhs.pdf  
- statsmodels OLS: https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html  
- OLS Results (attributes like `rsquared_adj`, `pvalues`): https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.html  

### Pseudocode Plan
1) **Load CSV** → preview columns/shape; confirm target & candidate predictors exist.  
2) **Assign Y + Xs** (start small, add features with a hypothesis). Coerce **just these columns** to numeric.  
3) **Light prep:** derive `trip_time_minutes` from `trip_time` (seconds); convert flags (`shared_request_flag`, `wav_request_flag`) to 0/1 if present.  
4) **Model sets (3 total):**  
   - **Model A (parsimonious).**  
   - **Model B (adds one meaningful predictor).**  
   - **Model C (adds 1–2 more, e.g., flags).  
5) **Add intercept** and **fit** each with OLS on the same rows.  
6) **Record metrics:** `rsquared_adj`, coefficient table, and **p-values**.  
7) **Compare:** Prefer higher **Adjusted R²** and keep an eye on **p-values** (and signs/units).  
8) **Interpretation:** Write unit-based sentences **holding others constant**.  
9) **Selection rationale:** Pick the simplest model that improves **Adjusted R²** and 


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

### Step 0 — Setup & Imports

In [2]:
import pandas as pd, numpy as np
import statsmodels.api as sm
from pathlib import Path

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

### Step 1 — Load CSV & Preview
- Point to your For Hire Vehicle Data 
- Print **shape** and **columns**.

**Hint: You may have to drop missing values and do a force coercion to make sure the variables stay numeric (other coding assignments may help)**

In [3]:
df = pd.read_csv('/Users/Marcy_Student/Downloads/FHV_072023 copy.csv')
print('Shape:', df.shape)
print('Columns:', df.columns)
df.head()

/var/folders/7n/rj4gbkk13_1f9n1qf54j12gh0000gp/T/ipykernel_1741/956305.py:1: DtypeWarning: Columns (11,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/Marcy_Student/Downloads/FHV_072023 copy.csv')


Shape: (8324591, 24)
Columns: Index(['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num',
       'request_datetime', 'on_scene_datetime', 'pickup_datetime',
       'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles',
       'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax',
       'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay',
       'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag',
       'wav_request_flag', 'wav_match_flag'],
      dtype='object')


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0005,B03406,NaN,07/01/2023 05:34:30 PM,NaN,07/01/2023 05:37:48 PM,07/01/2023 05:44:45 PM,158,68,1.2660,...,1.3500,2.7500,0.0000,2.0000,5.5700,N,N,N,N,False
1,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:53 PM,07/01/2023 05:37:15 PM,07/01/2023 05:55:15 PM,162,234,2.3500,...,1.5200,2.7500,0.0000,3.2800,13.3800,N,N,NaN,N,False
2,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:35:17 PM,07/01/2023 05:35:52 PM,07/01/2023 05:44:27 PM,161,163,0.8100,...,0.4900,2.7500,0.0000,0.0000,5.9500,N,N,NaN,N,False
3,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:37:39 PM,07/01/2023 05:39:35 PM,07/01/2023 06:23:02 PM,122,229,15.4700,...,5.1700,2.7500,0.0000,0.0000,54.4600,N,N,NaN,N,True
4,HV0003,B03404,B03404,07/01/2023 05:34:30 PM,07/01/2023 05:36:06 PM,07/01/2023 05:36:39 PM,07/01/2023 05:45:06 PM,67,14,1.5200,...,0.8500,0.0000,0.0000,3.0000,7.0100,N,N,NaN,N,False


### Step 2 —  Choose Target **Y** and Candidate Predictors

- Suggested **Y**: `base_passenger_fare` (USD).
- Start with **distance** and **time**; optionally add **flags** if present.
- Derive `trip_time_minutes` from `trip_time` (seconds) if available.

In [4]:
Y = 'base_passenger_fare'
candidate_numeric = ['trip_miles', 'trip_time']  
candidate_flags   = ['shared_request_flag', 'wav_request_flag']  

# Keep only columns we touch
keep_cols = [col for col in [Y, *candidate_numeric, *candidate_flags] if col in df.columns]
df = df[keep_cols].copy()

# Coerce numeric columns 
cols = ['base_passenger_fare', 'trip_time', 'trip_miles']
for c in cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.strip().str.replace(r'[^0-9.+\-eE]', '', regex=True),
        errors='coerce'
    )

# Trip_time from seconds to minutes
df['trip_time_minutes'] = pd.to_numeric(df['trip_time'], errors='coerce') / 60.0

# Convert flags to 0/1 if present (robust to 'Y'/'N', True/False, ints)
df[candidate_flags] = (
    df[candidate_flags]
    .replace({'Y': 1, 'y': 1, 'N': 0, 'n': 0, True: 1, False: 0})
    .fillna(0)
    .astype(int)
)


/var/folders/7n/rj4gbkk13_1f9n1qf54j12gh0000gp/T/ipykernel_1741/1766618987.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({'Y': 1, 'y': 1, 'N': 0, 'n': 0, True: 1, False: 0})


### Step 3 — Define Three Model Specs (A, B, C)
Example models you can chose any models you want as long as Model A has one term, Model B two terms, etc.

- **Model A:** distance only.  
- **Model B:** distance + time (minutes).  
- **Model C:** distance + time + flags (whichever exist).

In [ ]:
models = {}

# Model A: distance only.
models['A_dist'] = ['trip_miles']
# Model B: distance + time (minutes)B: distance + time
models['B_dist_time'] = ['trip_miles', 'trip_time_minutes']

# Model C: distance + time + flags (whichever exist)
flags_present = [c for c in ['shared_request_flag','wav_request_flag'] if c in df.columns]
if 'trip_miles' in df.columns and 'trip_time_minutes' in df.columns and flags_present:
    models['C_dist_time_flags'] = ['trip_miles', 'trip_time_minutes'] + flags_present



Model specs: {'A_dist': ['trip_miles'], 'B_dist_time': ['trip_miles', 'trip_time_minutes'], 'C_dist_time_flags': ['trip_miles', 'trip_time_minutes', 'shared_request_flag', 'wav_request_flag']}


In [6]:
# Model A (distance)
X_A = sm.add_constant(df[['trip_miles']]) 
y_A = df[Y] 

# Model B (distance + time)
X_B = sm.add_constant(df[['trip_miles', 'trip_time_minutes']])
y_B = df[Y]


# Model C (distance + time + flag)
optional_flags = [c for c in ['shared_request_flag', 'wav_request_flag'] if c in df.columns]

features_C = ['trip_miles', 'trip_time_minutes'] + optional_flags

if features_C:  # only create model if there is a flag
    X_C = sm.add_constant(df[features_C])
    y_C = df[Y]


### Step 4 — Fit Each Model (with intercept) and Collect Adjusted R² & p-values


In [10]:
# Model A (distance only)
X1 = sm.add_constant(df[['trip_miles']])
y1 = df[Y]

modelA = sm.OLS(y1, X1).fit()
print("Model A (Distance Only)")
print("R²:", round(modelA.rsquared, 3),
      "Adj R²:", round(modelA.rsquared_adj, 3))


# Model B (distance + time)
X2 = sm.add_constant(df[['trip_miles', 'trip_time_minutes']])
y2 = df[Y]

modelB = sm.OLS(y2, X2).fit()
print("Model B (Distance + Time)")
print("R²:", round(modelB.rsquared, 3),
      "Adj R²:", round(modelB.rsquared_adj, 3))


# Model C (distance + time + optional flags)
flags_present = [c for c in ['shared_request_flag', 'wav_request_flag'] if c in df.columns]
features_c = ['trip_miles', 'trip_time_minutes'] + flags_present

X3 = sm.add_constant(df[features_c])
y3 = df[Y]

modelC = sm.OLS(y3, X3).fit()
print("Model C (Distance + Time + Flags)")
print("R²:", round(modelC.rsquared, 3),
      "Adj R²:", round(modelC.rsquared_adj, 3))


Model A (Distance Only)
R²: 0.808 Adj R²: 0.808
Model B (Distance + Time)
R²: 0.843 Adj R²: 0.843
Model C (Distance + Time + Flags)
R²: 0.845 Adj R²: 0.845


### Step 5 — Inspect Full Summaries (coefficients, p-values, diagnostics)

- Print summaries for the top 1–2 models by **Adjusted R²**.
- Write **unit-based** interpretations “holding others constant.”

In [ ]:
# Summaries
modelA.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     base_passenger_fare   R-squared:                       0.845
Model:                             OLS   Adj. R-squared:                  0.845
Method:                  Least Squares   F-statistic:                 1.139e+07
Date:                 Tue, 11 Nov 2025   Prob (F-statistic):               0.00
Time:                         15:17:10   Log-Likelihood:            -2.9202e+07
No. Observations:              8324591   AIC:                         5.840e+07
Df Residuals:                  8324586   BIC:                         5.840e+07
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   3.4461      0.005    681.045      0.000       3.436       3.456
trip_miles              2.2026      0.001   2745.958      0.000       2.201       2.204
trip_time_minutes       0.4885      0.000   1374.062      0.000       0.488       0.489
shared_request_flag    -6.5582      0.019   -353.440      0.000      -6.595      -6.522
wav_request_flag       -1.3156      0.059    -22.483      0.000      -1.430      -1.201
==============================================================================
Omnibus:                 11278966.701   Durbin-Watson:                   1.944
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      67624031946.680
Skew:                           6.643   Prob(JB):                         0.00
Kurtosis:                     444.345   Cond. No.                         509.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [15]:
modelB.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     base_passenger_fare   R-squared:                       0.843
Model:                             OLS   Adj. R-squared:                  0.843
Method:                  Least Squares   F-statistic:                 2.237e+07
Date:                 Tue, 11 Nov 2025   Prob (F-statistic):               0.00
Time:                         15:18:12   Log-Likelihood:            -2.9264e+07
No. Observations:              8324591   AIC:                         5.853e+07
Df Residuals:                  8324588   BIC:                         5.853e+07
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 3.3276      0.005    654.154      0.000       3.318       3.338
trip_miles            2.2063      0.001   2730.971      0.000       2.205       2.208
trip_time_minutes     0.4856      0.000   1356.536      0.000       0.485       0.486
==============================================================================
Omnibus:                 11153723.298   Durbin-Watson:                   1.944
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      63858869427.114
Skew:                           6.499   Prob(JB):                         0.00
Kurtosis:                     431.880   Cond. No.                         44.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [14]:
modelC.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                            
===============================================================================
Dep. Variable:     base_passenger_fare   R-squared:                       0.845
Model:                             OLS   Adj. R-squared:                  0.845
Method:                  Least Squares   F-statistic:                 1.139e+07
Date:                 Tue, 11 Nov 2025   Prob (F-statistic):               0.00
Time:                         15:17:59   Log-Likelihood:            -2.9202e+07
No. Observations:              8324591   AIC:                         5.840e+07
Df Residuals:                  8324586   BIC:                         5.840e+07
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   3.4461      0.005    681.045      0.000       3.436       3.456
trip_miles              2.2026      0.001   2745.958      0.000       2.201       2.204
trip_time_minutes       0.4885      0.000   1374.062      0.000       0.488       0.489
shared_request_flag    -6.5582      0.019   -353.440      0.000      -6.595      -6.522
wav_request_flag       -1.3156      0.059    -22.483      0.000      -1.430      -1.201
==============================================================================
Omnibus:                 11278966.701   Durbin-Watson:                   1.944
Prob(Omnibus):                  0.000   Jarque-Bera (JB):      67624031946.680
Skew:                           6.643   Prob(JB):                         0.00
Kurtosis:                     444.345   Cond. No.                         509.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Step 6 — Interpretations (write below)

Using the **best model’s** coefficients interpret each coefficient using markdown

Model C

Distance: Each extra mile increases the base fare by about $2.20, holding trip time and ride type constant.

Time: Each additional minute of trip adds about $0.49 to the fare, holding distance and ride type constant.

Shared rides: Trips marked as shared are roughly $6.56 cheaper than non-shared trips, all else equal.

Wheelchair-accessible (WAV) rides: These trips are about $1.32 cheaper than standard rides, holding distance and time constant.

## We Share — Reflection & Wrap‑Up

Write **2 short paragraphs** and be specific:

1) **Which model (A/B/C) do you pick and why?**  
Reference **Adjusted R²** (higher is better when comparing models with different numbers of predictors) and the **p-values**/signs of key coefficients.

Model B, which only looks at distance and trip time, explains 84.3% of the variation in passenger fares, showing that longer trips cost more, but it doesn’t account for ride type. Models A and C both do a slightly better with an adjusted R² of 0.845 because they include flags for shared rides and wheelchair-accessible vehicle requests, which tend to lower fares. Model C is the best choice because it includes all the important factors while staying easy to interpret. It shows that fares go up with distance and time, but are lower for shared or accessible trips, making it the most accurate model for predicting base passenger fares.

2) **Business explanation:**  
Give a stakeholder-friendly summary in **units** (e.g., “+1 mile ≈ +$X in base fare, holding time constant”). If you added flags, explain their effect plainly. Mention any limitations (e.g., time vs distance confounding, missing columns).

Each additional mile increases the base fare by about $2.20, and each extra minute adds around $0.49, assuming ride type stays the same. Shared rides are roughly $6.56 cheaper than regular trips, and wheelchair-accessible (WAV) trips are $1.32 cheaper. Some limitations are that the model only predicts base fare and may slightly mix up the effects of distance and time since longer trips usually take more minutes. It also only uses the ride features available in the dataset.